Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Modeling a Peptide (Deca-alanine) using AmberTools tleap

This notebook demonstrates how to construct an $\alpha$-helix structure consisting of 10 alanine residues (deca-alanine) using tleap, a module of  [AmberTools](https://ambermd.org/AmberTools.php)

The main steps covered in this notebook are:
1. **Environment Setup:** Installing AmberTools and creating directories for input/output.
2. **Input Preparation:** Creating the tleap input file (.in).
3. **Execution:** Running tleap to generate parameter files (.prmtop, .inpcrd) and converting them to PDB format.


## Step 1: Environment Setup

### Step 1-1: Preparing AmberTools
In this notebook, we use the tleap module included in AmberTools for peptide modeling. If AmberTools is not yet installed, you can install it using the following steps:

```bash
conda create --name AmberTools        # Create a virtual environment
conda activate AmberTools             # Activate the environment
conda install conda-forge::ambertools # Install AmberTools
conda deactivate
```

After installation, set the environment variables by running `amber.sh` whenever you use AmberTools:
```bash
conda activate AmberTools
source $CONDA_PREFIX/amber.sh
    cd to some working folder, and run AmberTools programs
conda deactivate
```

For more details, please refer to the [Official Amber Website](https://ambermd.org/GetAmber.php).

### Step 1-2: Preparing the Working Directory

We then create an `input` directory for input files and an `output` directory for the results.

In [ ]:
import os

# Define directory paths
out_dir = "./output/01_modeling"
inp_dir = "./input/01_modeling"

# Create directories only if they do not exist
os.makedirs(out_dir, exist_ok=True)
os.makedirs(inp_dir, exist_ok=True)

## Step 2: Creating the tleap Input File

We will create an input script to run tleap.


**Key Settings:**
* `source`: Loads the force field parameters for proteins (ff14SB).
* `sequence`: Defines the amino acid sequence. Chemical caps (ACE: Acetyl group, NME: Methyl group) are added to both ends.
* `impose`: Specifies dihedral angles ($\phi, \psi$) to create an $\alpha$-helix structure.
  * $\phi$ (Phi): -57.0 degrees (Atoms: C-N-CA-C)
  * $\psi$ (Psi): -47.0 degrees (Atoms: N-CA-C-N)

In [ ]:
tleap_commands = f"""
# 1. Load the force field
source leaprc.protein.ff14SB

# 2. Create the sequence
#    ACE: N-terminal cap (Acetyl)
#    NME: C-terminal cap (N-methyl)
mol = sequence {{ ACE ALA ALA ALA ALA ALA ALA ALA ALA ALA ALA NME }}

# 3. Apply alpha-helix structure (Impose)
#    Target: Residues 2-11 (the alanine portion, excluding caps)
#    Phi(φ): C-N-CA-C = -57.0 degrees
#    Psi(ψ): N-CA-C-N = -47.0 degrees
impose mol {{ 2 3 4 5 6 7 8 9 10 11 }} {{ {{ "C" "N" "CA" "C" -57.0 }} {{ "N" "CA" "C" "N" -47.0 }} }}

# 4. Save parameter files (prmtop and inpcrd)
saveAmberParm mol {out_dir}/deca_alanine.prmtop {out_dir}/deca_alanine.inpcrd

# Exit tleap
quit
"""

# Write to file
filename = f"{inp_dir}/tleap_deca_alanine_helix.in"
with open(filename, "w") as f:
    f.write(tleap_commands)

## Step 3: Running tleap and Converting to PDB Format

We run `tleap` using the created input file, then convert the generated AMBER format files into PDB format.

* `tleap`: A tool for creating parameter files.
* `ambpdb`: A tool for creating PDB files from AMBER topology and coordinate files.

In [ ]:
# Run tleap
!tleap -f {inp_dir}/tleap_deca_alanine_helix.in > {out_dir}/tleap.log && mv leap.log {out_dir}

# Generate a PDB file using ambpdb
!ambpdb -p {out_dir}/deca_alanine.prmtop < {out_dir}/deca_alanine.inpcrd > {out_dir}/deca_alanine_helix.pdb

## Step 4: Verification
Let's verify that the files were generated correctly. It is successful if the following files are in the output directory:
* `deca_alanine.prmtop`: Force field parameter file
* `deca_alanine.inpcrd`: Coordinate file
* `deca_alanine_helix.pdb`: Structure file

## Next Step

The modeling of Deca-Alanine with an $\alpha$-helix structure is now complete.
In the next notebook [(02_equilibrium_nvt_md_en.ipynb)](./02_equilibrium_nvt_md_en.ipynb), we will perform Molecular Dynamics (MD) simulations to relax the modeled structure.